# SLM-WM 真实科学算子与仅图像盲检入口

该 Notebook 是当前主方法的正式 GPU 入口。它只以 `python -I` 调用统一宿主 launcher。launcher 创建精确 `workflow_orchestrator`, 再在一次会话中只准备一次 `sd35_method_runtime_gpu` 隔离子解释器, 再由该解释器运行分支风险、语义条件 Jacobian 低响应子空间、空间 LF 载体、幅值尾部载体、真实 Q/K 注意力几何、仅图像盲检、真实图像攻击、正式 Inception FID/KID 和可选正式消融。

Git 工作区、outputs 和隔离 Python 均在本地 `/content` 执行, 避免 Drive FUSE 承担大量小文件读取与可执行文件权限。每个完成 Prompt、正式消融运行和 Inception 特征 batch 都通过摘要绑定的 checkpoint 原子同步到 Google Drive。Colab 中断后重新执行全部单元格即可恢复; 未闭合 checkpoint 明确不具备论文证据资格。完成后的闭合结果包会独立镜像并可经 closure 校验直接恢复。


## 使用顺序

1. 首次直接使用默认 `probe_paper`, 完成 70 个 Prompt 的端到端验证。
2. 把 Colab 运行时设置为 GPU, 建议使用显存不低于 24GB 的 L4、A100 或同等级设备。
3. 在 Colab 密钥中配置 `HF_TOKEN`, 且账号已获得 SD3.5 Medium 权限。
4. `SLM_WM_MAX_NEW_PROMPTS_PER_SESSION` 控制单次会话新增 Prompt 数。会话结束或中断后可重复执行全部单元格。
5. 数据集主流程完成后, 再把 `SLM_WM_RUN_FORMAL_ABLATION` 改为 `True`。正式消融同样支持跨会话续跑。
6. `workflow_decision` 为 `resume_required` 只表示仍需续跑, 不表示实验失败。只有正式摘要和结果包齐备后才能进入论文结果闭合。

In [ ]:
SLM_WM_PAPER_RUN_NAME = "probe_paper"

import os
from pathlib import Path
import shutil

from google.colab import drive

drive.mount('/content/drive')
# 默认运行 probe_paper; 通过后再显式切换到 pilot_paper 或 full_paper.
SLM_WM_MAX_NEW_PROMPTS_PER_SESSION = 5
SLM_WM_MAX_NEW_ABLATION_RUNS_PER_SESSION = 5
SLM_WM_RUN_FORMAL_ABLATION = False

os.environ["SLM_WM_PAPER_RUN_NAME"] = SLM_WM_PAPER_RUN_NAME
os.environ["SLM_WM_MAX_NEW_PROMPTS_PER_SESSION"] = str(SLM_WM_MAX_NEW_PROMPTS_PER_SESSION)
os.environ["SLM_WM_MAX_NEW_ABLATION_RUNS_PER_SESSION"] = str(SLM_WM_MAX_NEW_ABLATION_RUNS_PER_SESSION)
os.environ.setdefault("SLM_WM_ENABLE_DIFFUSION_ATTACKS", "1")
os.environ.setdefault("SLM_WM_INCEPTION_DEVICE", "cuda")

# 模型缓存保存在 Drive; Git、outputs 和隔离环境均在本地 /content 执行。
drive_root = Path("/content/drive/MyDrive/SLM")
hf_home = drive_root / "model_cache" / "huggingface"
torch_home = drive_root / "model_cache" / "torch"
hf_home.mkdir(parents=True, exist_ok=True)
torch_home.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(hf_home))
os.environ.setdefault("TORCH_HOME", str(torch_home))
drive_usage = shutil.disk_usage(drive_root)
print({
    "paper_run_name": SLM_WM_PAPER_RUN_NAME,
    "hf_home": str(hf_home),
    "torch_home": str(torch_home),
    "drive_free_gib": round(drive_usage.free / 1024**3, 2),
})

In [ ]:
import os
import re
import subprocess
import sys
from pathlib import Path

repository_url = os.environ.get("SLM_WM_REPOSITORY_URL", "https://github.com/RICHAAARC/SLM-WM.git")
repository_commit = os.environ.get("SLM_WM_REPOSITORY_COMMIT", "")
if re.fullmatch(r"[0-9a-f]{40}", repository_commit) is None:
    raise ValueError("正式运行前必须通过 SLM_WM_REPOSITORY_COMMIT 提供精确40位小写 Git SHA")
workspace_dir = Path(os.environ.get("SLM_WM_WORKSPACE_DIR", f"/content/slm_wm_repository_{repository_commit[:12]}"))
workspace_dir.parent.mkdir(parents=True, exist_ok=True)
if workspace_dir.exists() and not (workspace_dir / ".git").is_dir():
    raise RuntimeError("SLM_WM_WORKSPACE_DIR 已存在但不是 Git 仓库")
if not (workspace_dir / ".git").is_dir():
    subprocess.run(["git", "clone", repository_url, str(workspace_dir)], check=True)

status_before_checkout = subprocess.run(
    ["git", "status", "--porcelain=v1", "--untracked-files=all"],
    cwd=workspace_dir,
    check=True,
    capture_output=True,
    text=True,
).stdout
if status_before_checkout:
    raise RuntimeError("复用 Colab 工作区前必须先清理 Git 工作树")
subprocess.run(["git", "fetch", "origin", "--tags", "--prune"], cwd=workspace_dir, check=True)
subprocess.run(["git", "rev-parse", "--verify", repository_commit + "^{commit}"], cwd=workspace_dir, check=True)
subprocess.run(["git", "checkout", "--detach", repository_commit], cwd=workspace_dir, check=True)

os.chdir(workspace_dir)
print({"workspace_dir": str(workspace_dir), "repository_commit": repository_commit})


In [ ]:
FORMAL_HOST_LAUNCHER = "scripts/run_formal_workflow_host.py"
print({"formal_host_launcher": FORMAL_HOST_LAUNCHER})


In [ ]:
import os

try:
    from google.colab import userdata
    token_from_secret = userdata.get("HF_TOKEN")
except Exception:
    token_from_secret = None

if token_from_secret and not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = token_from_secret
print({"hf_token_available_for_scientific_child": bool(os.environ.get("HF_TOKEN"))})


In [ ]:
# 论文运行配置由精确 workflow_orchestrator 子解释器统一加载.


In [ ]:
gpu_query = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total",
        "--format=csv,noheader,nounits",
    ],
    check=True,
    capture_output=True,
    text=True,
)
gpu_rows = [row.strip() for row in gpu_query.stdout.splitlines() if row.strip()]
if not gpu_rows:
    raise RuntimeError("该正式入口要求 Colab GPU 运行时")
device_name, total_memory_mib = [part.strip() for part in gpu_rows[0].rsplit(",", 1)]
device_report = {
    "cuda_available": True,
    "device_count": len(gpu_rows),
    "device_name": device_name,
    "total_memory_gib": round(float(total_memory_mib) / 1024, 2),
}
print(device_report)
assert device_report["total_memory_gib"] >= 20.0, "建议改用显存不低于 24GB 的 Colab GPU"


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

workflow_route = "mechanism_ablation" if SLM_WM_RUN_FORMAL_ABLATION else "image_only_dataset"
result_path = Path("outputs/formal_workflow_execution") / SLM_WM_PAPER_RUN_NAME / workflow_route / "workflow_result.json"
host_command = [
    sys.executable,
    "-I",
    FORMAL_HOST_LAUNCHER,
    "--root",
    ".",
    "--repository-commit",
    repository_commit,
    "gpu",
    "--workflow",
    workflow_route,
    "--paper-run-name",
    SLM_WM_PAPER_RUN_NAME,
    "--result-path",
    result_path.as_posix(),
]
subprocess.run(host_command, check=True)
formal_workflow_result = json.loads(result_path.read_text(encoding="utf-8"))
assert formal_workflow_result["decision"] == "pass", formal_workflow_result
formal_workflow_result


In [ ]:
print(json.dumps(formal_workflow_result, ensure_ascii=False, sort_keys=True, indent=2))
